In [ ]:
# Install dependencies (run once)
!pip install requests beautifulsoup4 pandas

In [1]:
import re
import time
import random
import requests
from bs4 import BeautifulSoup
import pandas as pd

BASE = "https://krisha.kz"
# Sale of apartments in Karaganda
LIST_URL = "https://krisha.kz/prodazha/kvartiry/karaganda/"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "ru-RU,ru;q=0.9,en;q=0.8",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}

session = requests.Session()
session.headers.update(HEADERS)

In [2]:
def clean_int(text):
    """Extract digits from a string and return an int, or None."""
    if not text:
        return None
    digits = re.sub(r"[^\d]", "", text)
    return int(digits) if digits else None


def parse_title(title):
    """Parse 'rooms' and 'area_m2' from a card title.

    Example title: '2-комнатная квартира · 71 м²'
    """
    rooms = area = None
    if not title:
        return rooms, area

    m = re.search(r"(\d+)-комнатн", title)
    if m:
        rooms = int(m.group(1))

    m = re.search(r"([\d.,]+)\s*м²", title)
    if m:
        try:
            area = float(m.group(1).replace(",", "."))
        except ValueError:
            area = None

    return rooms, area


def parse_card(card):
    """Extract one apartment record from a krisha .a-card element."""
    title_el = card.select_one("a.a-card__title")
    if title_el is None:
        return None

    title = title_el.get_text(strip=True)
    href = title_el.get("href", "")
    url = BASE + href if href.startswith("/") else href

    price_el = card.select_one(".a-card__price")
    price = clean_int(price_el.get_text()) if price_el else None

    # .a-card__subtitle = district / street ; .a-card__stats = city + date
    addr_el = card.select_one(".a-card__subtitle")
    address = addr_el.get_text(" ", strip=True) if addr_el else None

    stats_el = card.select_one(".a-card__stats")
    stats = stats_el.get_text(" ", strip=True) if stats_el else None

    rooms, area = parse_title(title)
    price_per_m2 = round(price / area) if (price and area) else None

    return {
        "id": card.get("data-id") or card.get("data-list-id"),
        "title": title,
        "rooms": rooms,
        "area_m2": area,
        "price_tenge": price,
        "price_per_m2": price_per_m2,
        "address": address,
        "city_date": stats,
        "url": url,
    }


def scrape_page(page):
    """Return a list of apartment records from one listing page."""
    params = {} if page == 1 else {"page": page}
    resp = session.get(LIST_URL, params=params, timeout=30)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")

    cards = soup.select("div.a-card[data-id]") or soup.select("div.a-card")
    records = []
    for card in cards:
        rec = parse_card(card)
        if rec and rec["title"]:
            records.append(rec)
    return records

In [3]:
# Collect listings until we reach TARGET rows (de-duplicated by id)
TARGET = 200       # 200 rows is enough and keeps the run fast
MAX_PAGES = 100    # safety cap

all_records = {}
page = 1

while len(all_records) < TARGET and page <= MAX_PAGES:
    try:
        records = scrape_page(page)
    except requests.HTTPError as e:
        print(f"Page {page}: HTTP error {e} -- stopping.")
        break

    if not records:
        print(f"Page {page}: no listings found -- reached the end.")
        break

    for rec in records:
        all_records[rec["id"] or rec["url"]] = rec

    print(f"Page {page:>3}: +{len(records):>2} cards | total unique: {len(all_records)}")
    page += 1
    time.sleep(random.uniform(1.0, 2.5))   # be polite, avoid rate limits

print(f"\nDone. Collected {len(all_records)} unique listings.")

Page   1: +21 cards | total unique: 21
Page   2: +20 cards | total unique: 41
Page   3: +20 cards | total unique: 60
Page   4: +20 cards | total unique: 80
Page   5: +20 cards | total unique: 100
Page   6: +20 cards | total unique: 120
Page   7: +20 cards | total unique: 140
Page   8: +20 cards | total unique: 160
Page   9: +20 cards | total unique: 180
Page  10: +20 cards | total unique: 200

Done. Collected 200 unique listings.


In [4]:
# Build the DataFrame and trim to TARGET rows
df = pd.DataFrame(list(all_records.values())).head(TARGET).reset_index(drop=True)

print(f"{len(df)} rows, {df.shape[1]} columns")
df.head(10)

200 rows, 9 columns


,id,title,rooms,area_m2,price_tenge,price_per_m2,address,city_date,url
0,693146384,1-комнатная квартира · 54.5 м²,1,54.5,30520000,560000,"Казыбек би р-н, Ержанова 63/2",Караганда 16 июн.,https://krisha.kz/a/show/693146384
1,693146389,3-комнатная квартира · 106.1 м²,3,106.1,59416000,560000,"Казыбек би р-н, Ержанова 63/2",Караганда 16 июн.,https://krisha.kz/a/show/693146389
2,693146382,2-комнатная квартира · 86.7 м²,2,86.7,48552000,560000,"Казыбек би р-н, Ержанова 63/2",Караганда 16 июн.,https://krisha.kz/a/show/693146382
3,1012319331,1-комнатная квартира · 30.5 м² · 5/5 этаж,1,30.5,15500000,508197,"Казыбек би р-н, мкр Новый Город, Ержанова 30",Караганда 16 июн.,https://krisha.kz/a/show/1012319331
4,693146383,2-комнатная квартира · 87.4 м²,2,87.4,48944000,560000,"Казыбек би р-н, Ержанова 63/2",Караганда 16 июн.,https://krisha.kz/a/show/693146383
5,693146387,3-комнатная квартира · 96.5 м²,3,96.5,54040000,560000,"Казыбек би р-н, Ержанова 63/2",Караганда 16 июн.,https://krisha.kz/a/show/693146387
6,1010848799,4-комнатная квартира · 100 м² · 6/17 этаж,4,100.0,55000000,550000,"Казыбек би р-н, мкр Городской Аэропорт, Букето...",Караганда 16 июн.,https://krisha.kz/a/show/1010848799
7,1011820130,2-комнатная квартира · 47 м² · 4/5 этаж,2,47.0,20900000,444681,"Казыбек би р-н, ул. Гоголя 49",Караганда 16 июн.,https://krisha.kz/a/show/1011820130
8,1012988997,3-комнатная квартира · 89 м² · 1/5 этаж,3,89.0,37500000,421348,"Казыбек би р-н, мкр Юго-Восток, Таттимбета 5/2",Караганда 16 июн.,https://krisha.kz/a/show/1012988997
9,1013145836,2-комнатная квартира · 63.5 м² · 1/19 этаж,2,63.5,51900000,817323,"Казыбек би р-н, мкр Новый Город, Мкр Байкена А...",Караганда 16 июн.,https://krisha.kz/a/show/1013145836


In [5]:
# --- Add 4 features from each apartment's detail page ---
# These characteristics are not on the listing cards, so we open every /a/show/<id> page.
# Map krisha's data-name keys -> our English column names.
DETAIL_FIELDS = {
    "flat.balcony":   "balcony",         # Балкон
    "flat.parking":   "parking",         # Парковка
    "live.furniture": "furnished",       # Квартира меблирована
    "ceiling":        "ceiling_height",  # Высота потолков
}


def get_details(url):
    """Fetch one detail page and return the 4 target characteristics."""
    details = {col: None for col in DETAIL_FIELDS.values()}
    try:
        resp = session.get(url, timeout=30)
        resp.raise_for_status()
    except requests.RequestException:
        return details

    soup = BeautifulSoup(resp.text, "html.parser")
    for item in soup.select("div.offer__info-item"):
        col = DETAIL_FIELDS.get(item.get("data-name"))
        if col:
            value = item.select_one(".offer__advert-short-info")
            if value:
                details[col] = value.get_text(strip=True)
    return details


# Fetch detail pages for every listing (one request per row)
detail_rows = []
for n, url in enumerate(df["url"], start=1):
    detail_rows.append(get_details(url))
    if n % 25 == 0 or n == len(df):
        print(f"{n:>4}/{len(df)} detail pages fetched")
    time.sleep(random.uniform(0.4, 1.0))   # be polite

df = pd.concat([df, pd.DataFrame(detail_rows, index=df.index)], axis=1)

print("\nNew columns added:", list(DETAIL_FIELDS.values()))
df[["title", "balcony", "parking", "furnished", "ceiling_height"]].head(10)

  25/200 detail pages fetched
  50/200 detail pages fetched
  75/200 detail pages fetched
 100/200 detail pages fetched
 125/200 detail pages fetched
 150/200 detail pages fetched
 175/200 detail pages fetched
 200/200 detail pages fetched

New columns added: ['balcony', 'parking', 'furnished', 'ceiling_height']


,title,balcony,parking,furnished,ceiling_height
0,1-комнатная квартира · 54.5 м²,None,None,None,3 м
1,3-комнатная квартира · 106.1 м²,None,None,None,3 м
2,2-комнатная квартира · 86.7 м²,None,None,None,3 м
3,1-комнатная квартира · 30.5 м² · 5/5 этаж,None,None,None,None
4,2-комнатная квартира · 87.4 м²,None,None,None,3 м
5,3-комнатная квартира · 96.5 м²,None,None,None,3 м
6,4-комнатная квартира · 100 м² · 6/17 этаж,None,None,None,None
7,2-комнатная квартира · 47 м² · 4/5 этаж,None,None,полностью,None
8,3-комнатная квартира · 89 м² · 1/5 этаж,None,None,None,3 м
9,2-комнатная квартира · 63.5 м² · 1/19 этаж,лоджия,None,None,None


In [6]:
# --- Save the final dataset to a CSV file ---
out_path = "karaganda_apartments.csv"
df.to_csv(out_path, index=False, encoding="utf-8-sig")  # utf-8-sig opens cleanly in Excel

print(f"Saved {df.shape[0]} rows and {df.shape[1]} columns to {out_path}")
print("Columns:", list(df.columns))

Saved 200 rows and 13 columns to karaganda_apartments.csv
Columns: ['id', 'title', 'rooms', 'area_m2', 'price_tenge', 'price_per_m2', 'address', 'city_date', 'url', 'balcony', 'parking', 'furnished', 'ceiling_height']
